# Primary vs Metastatic Metabolic Signaling Analysis

This notebook performs differential expression of metabolic target genes between primary and metastatic sites for a given cancer type, generating a volcano plot and an interactive table.
It also integrates LIANA+ Cell-Cell Communication results to see which target genes are actively mediating signaling.

**Prerequisite:** Run `cancer_cellxgene_integration.ipynb` to generate the `.h5ad` and `_cellxgene_liana_results.csv` outputs first.

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import os
import glob
import matplotlib.pyplot as plt
import seaborn as sns
from plotly import express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

sc.settings.verbosity = 2

## 1. Configuration

Select the primary and metastatic base prefixes generated by `cancer_cellxgene_integration.ipynb`. We will load both `.h5ad` and LIANA+ `.csv` files using these prefixes to avoid overwriting.

In [ ]:
# Base directories
BASE_DIR = '../'
DATA_DIR = os.path.join(BASE_DIR, 'input', 'databases')
OUTPUT_DIR = os.path.join(BASE_DIR, 'output')

# --- CONFIGURATION ---
# The filename prefixes generated by cancer_cellxgene_integration.ipynb (without the extension)
PRIMARY_PREFIX = 'breast_100k_whole_transcriptome_2025-11-08'
META_PREFIX = 'liver_skeletal-system-bone_brain_100k_whole_transcriptome_2025-11-08'
CANCER_TYPE_NAME = 'breast_cancer' # Used for naming the final output file
# ---------------------

primary_h5ad_file = os.path.join(OUTPUT_DIR, f'{PRIMARY_PREFIX}.h5ad')
meta_h5ad_file = os.path.join(OUTPUT_DIR, f'{META_PREFIX}.h5ad')

primary_liana_file = os.path.join(OUTPUT_DIR, f'{PRIMARY_PREFIX}_cellxgene_liana_results.csv')
meta_liana_file = os.path.join(OUTPUT_DIR, f'{META_PREFIX}_cellxgene_liana_results.csv')

# Load the MetabConnectomeDB unique pairs to get target genes
metab_db_path = os.path.join(OUTPUT_DIR, 'human_database_merge_unique_metab_target_pairs_with_HMDB_Info.csv')

## 2. Load Metabolic Target Vocabulary

In [ ]:
# Load the MetabConnectomeDB target genes
df_metab = pd.read_csv(metab_db_path)
df_metab['Target'] = df_metab['Target'].astype(str).str.split(r'[,;]')
df_metab = df_metab.explode('Target')
df_metab['Target'] = df_metab['Target'].str.strip()
df_metab = df_metab[df_metab['Target'] != 'nan'].dropna(subset=['Target'])
target_genes = set(df_metab['Target'].unique())
print(f"Loaded {len(target_genes):,} unique metabolic target genes from MetabConnectomeDB.")

## 3. Load LIANA+ Cell-Cell Communication Data

In [ ]:
primary_liana_targets = set()
meta_liana_targets = set()

if os.path.exists(primary_liana_file):
    df_liana_prim = pd.read_csv(primary_liana_file)
    primary_liana_targets = set(df_liana_prim['receptor_complex'].dropna().unique()).union(set(df_liana_prim['ligand_complex'].dropna().unique()))
    print(f"Loaded Primary LIANA results: {len(primary_liana_targets)} genes involved in signaling.")
else:
    print(f"⚠️ Primary LIANA file not found: {primary_liana_file}")

if os.path.exists(meta_liana_file):
    df_liana_meta = pd.read_csv(meta_liana_file)
    meta_liana_targets = set(df_liana_meta['receptor_complex'].dropna().unique()).union(set(df_liana_meta['ligand_complex'].dropna().unique()))
    print(f"Loaded Meta LIANA results: {len(meta_liana_targets)} genes involved in signaling.")
else:
    print(f"⚠️ Meta LIANA file not found: {meta_liana_file}")

## 4. Load and Merge scRNA-seq Data

In [ ]:
primary_list = globals().get('PRIMARY_TISSUES', ['breast'])
primary_name = primary_list[0].capitalize() if primary_list else 'Primary'

# Load single combined h5ad
print(f"Loading combined h5ad: {os.path.basename(h5ad_path)}")
adata = sc.read_h5ad(h5ad_path)

# Split by tissue
primary_mask = adata.obs['tissue_general'].isin(primary_list)
adata.obs['site'] = primary_mask.map({True: primary_name, False: 'Metastasis'})

adata_prim = adata[primary_mask].copy()
adata_meta = adata[~primary_mask].copy()

print(f"Merged dataset shape: {adata.shape}")
print(f"Primary cells: {primary_mask.sum()} | Metastatic cells: {(~primary_mask).sum()}")

## 5. Perform Differential Expression (Metastasis vs Primary)

In [ ]:
# We use Wilcoxon rank-sum test to compare Metastasis vs Primary
print("Running Differential Expression...")
sc.tl.rank_genes_groups(adata, groupby='site', groups=['Metastasis'], reference=primary_name, method='t-test')

# Extract DE results
result = adata.uns['rank_genes_groups']

de_df = pd.DataFrame({
    'names': result['names']['Metastasis'],
    'scores': result['scores']['Metastasis'],
    'logfoldchanges': result['logfoldchanges']['Metastasis'],
    'pvals': result['pvals']['Metastasis'],
    'pvals_adj': result['pvals_adj']['Metastasis']
})

# Calculate -log10 p-value for Volcano plot
min_pval = de_df[de_df['pvals_adj'] > 0]['pvals_adj'].min()
de_df['pvals_adj_safe'] = de_df['pvals_adj'].replace(0, min_pval)
de_df['-log10(p_adj)'] = -np.log10(de_df['pvals_adj_safe'])

# Filter for Metabolic Target Genes
metab_de_df = de_df[de_df['names'].isin(target_genes)].copy()

# Add categories for plotting
metab_de_df['Significance'] = 'Not Significant'
metab_de_df.loc[(metab_de_df['pvals_adj'] < 0.05) & (metab_de_df['logfoldchanges'] > 0.5), 'Significance'] = 'Up in Metastasis'
metab_de_df.loc[(metab_de_df['pvals_adj'] < 0.05) & (metab_de_df['logfoldchanges'] < -0.5), 'Significance'] = f"Up in {primary_name}"

print(f"Total Metabolic Target Genes analyzed: {len(metab_de_df)}")
print(metab_de_df['Significance'].value_counts())

## 6. Volcano Plot

In [ ]:
# Interactive Volcano Plot using Plotly
color_map = {
    'Not Significant': 'grey',
    'Up in Metastasis': 'red',
    f'Up in {primary_name}': 'blue'
}

fig = px.scatter(
    metab_de_df,
    x='logfoldchanges',
    y='-log10(p_adj)',
    color='Significance',
    hover_name='names',
    hover_data=['pvals_adj', 'logfoldchanges'],
    color_discrete_map=color_map,
    title=f"Volcano Plot: Metastatic vs Primary (Metabolic Target Genes - {CANCER_TYPE_NAME})",
    labels={'logfoldchanges': 'Log2 Fold Change (Metastasis vs Primary)', '-log10(p_adj)': '-Log10(Adjusted P-Value)'}
)

fig.add_vline(x=0.5, line_dash="dash", line_color="black")
fig.add_vline(x=-0.5, line_dash="dash", line_color="black")
fig.add_hline(y=-np.log10(0.05), line_dash="dash", line_color="black")

fig.update_layout(width=900, height=700)

# Save plotly as HTML
plotly_html_path = os.path.join(OUTPUT_DIR, f'primary_vs_metastatic_volcano_{CANCER_TYPE_NAME}.html')
fig.write_html(plotly_html_path)
print(f"Interactive volcano plot saved to: {plotly_html_path}")
fig.show()

# Static Seaborn plot with repel-style labels
from adjustText import adjust_text

plt.figure(figsize=(12, 9))
sns.scatterplot(
    data=metab_de_df,
    x='logfoldchanges',
    y='-log10(p_adj)',
    hue='Significance',
    palette=color_map,
    alpha=0.7
)

plt.axvline(0.5, ls='--', color='black', alpha=0.5)
plt.axvline(-0.5, ls='--', color='black', alpha=0.5)
plt.axhline(-np.log10(0.05), ls='--', color='black', alpha=0.5)

top_genes = metab_de_df[metab_de_df['Significance'] != 'Not Significant'].sort_values('pvals_adj').head(20)
texts = [plt.text(row['logfoldchanges'], row['-log10(p_adj)'], row['names'], fontsize=8)
         for _, row in top_genes.iterrows()]
adjust_text(texts, arrowprops=dict(arrowstyle='-', color='grey', lw=0.5))

plt.title(f"Metastasis vs Primary (Metabolic Targets - {CANCER_TYPE_NAME})")
plt.xlabel("Log2 Fold Change")
plt.ylabel("-Log10(Adj P-Value)")
plt.tight_layout()
plt.show()

## 7. Results Table Export with LIANA+ Annotations

In [ ]:
# Annotate whether the gene appears in the LIANA communication networks
def get_liana_status(gene):
    in_prim = gene in primary_liana_targets
    in_meta = gene in meta_liana_targets
    if in_prim and in_meta: return 'Both'
    if in_prim: return 'Primary Only'
    if in_meta: return 'Metastasis Only'
    return 'None'

metab_de_df['LIANA_Active_Network'] = metab_de_df['names'].apply(get_liana_status)

# Merge with our metabolite database
df_metab_summary = df_metab.groupby('Target').agg({
    'Metabolite_Name': lambda x: '; '.join(set(x.dropna().astype(str))),
    'HMDB_ID': lambda x: '; '.join(set(x.dropna().astype(str)))
}).reset_index()

results_table = metab_de_df.merge(df_metab_summary, left_on='names', right_on='Target', how='left')
results_table = results_table.drop(columns=['Target', 'pvals_adj_safe'])
results_table = results_table.sort_values('pvals_adj')

# Display the top significant targets
from IPython.display import display, HTML
display(HTML(results_table.head(30).to_html()))

# Save to a dynamically named CSV to prevent overwriting
output_csv = os.path.join(OUTPUT_DIR, f'primary_vs_metastasis_{CANCER_TYPE_NAME}_DE_metabolic_targets.csv')
results_table.to_csv(output_csv, index=False)
print(f"Results table saved to: {output_csv}")

## 8. Site-Specific Metastatic Convergence (Niche Comparison)

If the metastatic dataset contains cells from multiple distinct sites (e.g., Liver vs Brain vs Bone), we will automatically separate them and run DE for each site independently against the primary tumor. An UpSetPlot will be generated to visualize which metabolic target genes are universally upregulated across all metastatic sites (conserved signature) versus those that are unique to a specific niche.

In [ ]:
# Check if we have multiple metastatic sites
meta_sites = [t for t in adata_meta.obs['tissue_general'].unique() if pd.notnull(t)]

if len(meta_sites) > 1:
    print(f"Found multiple metastatic niches: {meta_sites}\nRunning site-specific DE...")
    
    # We need UpSetPlot for visualization
    try:
        from upsetplot import from_contents, plot
    except ImportError:
        import subprocess
        print("Installing upsetplot...")
        subprocess.check_call(['pip', 'install', 'upsetplot'])
        from upsetplot import from_contents, plot
        
    site_significant_genes = {}
    
    for site in meta_sites:
        # Create a subset with just the primary cells and THIS specific metastatic site
        # We assume primary cells are likely labeled under tissues like 'breast', 'breast gland', 'lung', 'colon' etc., 
        # but we can cleanly filter using adata.obs['site']
        
        adata_site = adata[(adata.obs['tissue_general'] == site) | (adata.obs['site'] == primary_name)].copy()
        adata_site.obs['comparison_group'] = adata_site.obs.apply(lambda x: site if x['site'] == 'Metastasis' else primary_name, axis=1)
        
        try:
            sc.tl.rank_genes_groups(adata_site, groupby='comparison_group', groups=[site], reference=primary_name, method='t-test')
            result = adata_site.uns['rank_genes_groups']
            
            # Filter for significant metabolic targets
            site_df = pd.DataFrame({
                'names': result['names'][site],
                'logfoldchanges': result['logfoldchanges'][site],
                'pvals_adj': result['pvals_adj'][site]
            })
            
            # Significant up-regulated in metastasis
            sig_up = site_df[(site_df['names'].isin(target_genes)) & 
                             (site_df['pvals_adj'] < 0.05) & 
                             (site_df['logfoldchanges'] > 0.5)]['names'].tolist()
            
            site_significant_genes[site] = sig_up
            print(f" - {site}: {len(sig_up)} upregulated metabolic targets")
            
        except Exception as e:
            print(f" - {site}: Could not run DE (maybe not enough cells). Error: {e}")
            
    if site_significant_genes:
        plt.figure(figsize=(10, 6))
        upset_data = from_contents(site_significant_genes)
        upset_axes = plot(upset_data, show_counts=True, sort_by='cardinality')
        for txt in upset_axes['intersections'].texts:
            txt.set_fontsize(6)
        plt.title(f"Metastatic Convergence: Upregulated Targets across Niches")
        plt.show()
        
        # Output the intersection (the pan-metastatic signature)
        if len(site_significant_genes) >= 2:
            common_genes = set.intersection(*[set(g) for g in site_significant_genes.values()])
            print(f"\n🔥 Pan-Metastatic Conserved Targets ({len(common_genes)} genes):")
            print(list(common_genes))
else:
    print(f"Only one or zero metastatic sites found ({meta_sites}). Site-specific comparison skipped.")


## Automated HTML Export

Generate a styled HTML report of this notebook for sharing and viewing.

In [ ]:
import subprocess
import sys
import os

notebook_filename = 'primary_vs_metastasis_comparison.ipynb'

# remove extension for nbconvert
output_base = output_csv.replace(".csv", "")

print(f"Executing full notebook HTML export for '{notebook_filename}'...")

# Get Jupyter binary
jupyter_bin = os.path.join(os.path.dirname(sys.executable), 'jupyter')
if not os.path.exists(jupyter_bin):
    jupyter_bin = 'jupyter'

cmd_html = [
    jupyter_bin,
    "nbconvert",
    "--to", "html",
    notebook_filename,
    "--output", output_base
]

res_html = subprocess.run(cmd_html, capture_output=True, text=True)

if res_html.returncode == 0:
    print("🎉 SUCCESS: Notebook successfully exported as a styled HTML report!")
    print(f"   -> Saved to: '{output_base}.html'")
else:
    print("❌ HTML export failed.")
    print(res_html.stderr)
